# Extractor test — PerfMnS key sections

Runs Reader → ContractBuilder, then calls **only** the `extractor_node` on the
seven PerfMnS sections that carry the service's rules, so we can inspect the
extraction in isolation — fast, no Validator/Reflector loop.

Checks the errors found earlier against the official `TS28532_PerfMnS.yaml`:
1. resource path is the real URI (`/notificationSink`), never the notification name;
2. schema name/casing is canonical (`NotifyThresholdCrossing`);
3. inherited `NotificationHeader` fields are not re-emitted as own properties;
4. the enumerations (`PerfNotificationTypes`, `PerfMetricDirection`) are extracted.

Requires a running Qdrant on localhost:6333 and OPENAI_API_KEY in .env.

In [ ]:
# Step 1 — Imports
from openapi_rulesbank.config import get_logger
from openapi_rulesbank.config.paths import TSPEC_DATA_TEST_FILE
from openapi_rulesbank.nodes.reader import reader_node
from openapi_rulesbank.nodes.contract_builder import contract_builder_node
from openapi_rulesbank.nodes.extractor import extractor_node
from openapi_rulesbank.tools.rag_tools import drop_target_spec_collection

TARGET_SERVICE = 'PerfMnS'
logger = get_logger(__name__)
logger.info('Imports OK.')

In [ ]:
# Step 2 — Reader + ContractBuilder (gives the Extractor its scope + spec RAG)
reader_result = reader_node({
    'main_doc_path':       TSPEC_DATA_TEST_FILE,
    'auxiliary_doc_paths': [],
    'target_yaml':         TARGET_SERVICE,
})
sections = reader_result['parsed_sections']

contract_result = contract_builder_node({
    'parsed_sections': sections,
    'target_yaml':     TARGET_SERVICE,
})
contract        = contract_result['service_contract']
spec_collection = contract_result.get('spec_collection_name', '')
print('stage3_clause:', contract.get('stage3_clause'))
print('spec collection:', spec_collection)

In [ ]:
# Step 3 — Pick the key sections and build a minimal extraction plan, IN DOCUMENT
# ORDER. The Extractor reads extraction_plan.sections_to_extract[current_section_idx],
# so we assemble the plan in clause order and iterate the index — the 'General'
# clause (which declares the imported NotificationHeader) must come before the
# 'Type' clause that embeds it, so the reference memory is populated in time.
KEY_PREFIXES = [
    '12.3.1.2.2.2',        # Notification notifyThresholdCrossing (mapping)
    '12.3.1.2.3.2.1 ',     # Resource "/notificationSink"
    '12.3.1.2.3.2.1.2',    # URI
    '12.3.1.2.3.2.1.3.1',  # POST
    '12.3.1.2.4.1',        # General — declares 'Data types imported' (NotificationHeader → TS 28.623)
    '12.3.1.2.4.2.1',      # Type NotifyThresholdCrossing (embeds NotificationHeader)
    '12.3.1.2.4.6.3',      # Enumeration PerfNotificationTypes
    '12.3.1.2.4.6.4',      # Enumeration PerfMetricDirection
]

def pick(prefix):
    return next((s for s in sections if s['title'].startswith(prefix)), None)

key_sections = [s for p in KEY_PREFIXES if (s := pick(p))]
plan = {'sections_to_extract': [
    {'section_id': s['section_id'], 'title': s['title'],
     'priority': 'high', 'extraction_focus': 'Extract all concrete rules this section defines.',
     'notes': ''}
    for s in key_sections
]}
print(f'{len(key_sections)} key section(s):')
for s in key_sections:
    print(f"  [{s['section_id']}] {s['title'][:55]}")

In [ ]:
# Step 4 — Run the Extractor on each section (first attempt, no correction loop).
# We thread other_spec_ref_memory across sections by hand, mimicking the graph's
# merge reducer, so a reference declared in an earlier section (e.g. the 'General'
# clause) is available to the later 'Type' clause that uses it.
base_state = {
    **reader_result,
    'service_contract':      contract,
    'spec_collection_name':  spec_collection,
    'extraction_plan':       plan,
    'helper_context':        '',
    'validation_errors':     [],
    'validator_section_reflection': [],
}

all_rules = []
ref_memory = {}
for idx in range(len(key_sections)):
    out = extractor_node({
        **base_state,
        'current_section_idx': idx,
        'section_iteration_count': 0,
        'other_spec_ref_memory': ref_memory,
    })
    all_rules.extend(out['raw_rules'])
    ref_memory = {**ref_memory, **out.get('other_spec_ref_memory', {})}  # merge, like the reducer
print(f'Extracted {len(all_rules)} raw rule(s) across {len(key_sections)} section(s).')
print('other_spec_ref_memory accumulated:', ref_memory)

In [ ]:
# Step 5 — Inspect every rule (type | name | object | field)
for i, r in enumerate(all_rules):
    m = r.get('openapi_mapping', {})
    print(f"{i+1:>2}. sec{r['section_id']} [{r['rule_type']:<15}] {r['source_name']:<28} "
          f"obj={m.get('openapi_object','')!r} field={m.get('openapi_field','')!r}")

In [ ]:
# Step 6 — Check against the four known error classes
INHERITED = {'objectClass', 'objectInstance', 'notificationId', 'notificationType',
             'eventTime', 'systemDN', 'href'}

paths      = {m.get('openapi_object','') for r in all_rules if r['rule_type'] in ('path_operation','request_body','response') for m in [r.get('openapi_mapping',{})]}
schema_objs = {m.get('openapi_object','') for r in all_rules if r['rule_type']=='schema_property' for m in [r.get('openapi_mapping',{})]}
props       = {r['source_name'] for r in all_rules if r['rule_type']=='schema_property'}

print('1. Path fabricated from notification name?')
bad = [p for p in paths if 'notifyThresholdCrossing' in p or 'notifythresholdcrossing' in p.lower()]
print('   FAIL, fabricated paths:', bad if bad else '   none — OK')
print('   real path present (/notificationSink)?', any('notificationSink' in p for p in paths))

print('\n2. Schema name casing (should be NotifyThresholdCrossing, capital N)')
print('   schema objects seen:', sorted(schema_objs))

print('\n3. Inherited NotificationHeader fields leaked as own properties?')
leaked = sorted(props & INHERITED)
print('   leaked:', leaked if leaked else '   none — OK')

print('\n4. Enumerations extracted?')
print('   PerfNotificationTypes:', any('PerfNotificationTypes' in o for o in schema_objs))
print('   PerfMetricDirection  :', any('PerfMetricDirection' in o for o in schema_objs))

In [ ]:
# Step 7 — Clean up the temporary spec collection
if spec_collection:
    drop_target_spec_collection(spec_collection)
    print('dropped', spec_collection)